In [1]:
from datasets import load_dataset

dataset = load_dataset("imdb")


In [2]:
import pandas as pd
import numpy as np
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.shape)
print(train_df.head(2))
print(train_df["label"].value_counts())  

(25000, 2)
                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
label
0    12500
1    12500
Name: count, dtype: int64


In [4]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [5]:
train_df["text_len"] = train_df["text"].str.len()

In [6]:
print(train_df["text_len"].describe())
print(train_df["text"].iloc[0]) 

count    25000.00000
mean      1325.06964
std       1003.13367
min         52.00000
25%        702.00000
50%        979.00000
75%       1614.00000
max      13704.00000
Name: text_len, dtype: float64
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher,

In [7]:
from bs4 import BeautifulSoup

In [8]:
train_df["text"] = train_df["text"].apply(
    lambda x: BeautifulSoup(x,"html.parser").get_text()
)

In [9]:
test_df["text"] = test_df["text"].apply(
    lambda x: BeautifulSoup(x,"html.parser").get_text()
)

In [10]:
print(train_df["text"].iloc[0])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot lik

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
tfidf = TfidfVectorizer(max_features=10000)
x_train = tfidf.fit_transform(train_df["text"])
x_test = tfidf.transform(test_df["text"])
y_train = train_df[["label"]].values.ravel()
y_test = test_df[["label"]].values.ravel()

In [13]:
y_train

array([0, 0, 0, ..., 1, 1, 1], shape=(25000,))

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *

In [15]:
LR = LogisticRegression()

model_lr = LR.fit(x_train,y_train)

y_pred = model_lr.predict(x_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [16]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
joblib.dump(model_lr, "models/lr_model.pkl")

['models/lr_model.pkl']